# 🎬 Video Object Replacement Pipeline v3

Ноутбук работает из корня `video_changer/` и использует модули `src/`.

## Диффузия (выбор `API_MODE` в конфиге)
- **`wan`** — **Replicate** + видео-модель WAN inpaint (по умолчанию `andreasjansson/wan-1.3b-inpaint`). Нужен **`REPLICATE_API_TOKEN`**.
- **`nano`** — **Gen-API Nano Banana 2** (как `video_changer_colab.ipynb`). Нужен **`GEN_API_KEY`**.

## Остальное
- Shot cut detection, CLIP Re-ID, bbox crop, опционально RAFT flow, regenerate vs warp.

## Веса локально
`models/`: `sam2_hiera_large.pt`, `groundingdino_swint_ogc.pth`. SAM2 config: `configs/sam2.1/sam2.1_hiera_l.yaml`.


## 1. Установка зависимостей

Используем `requirements.txt`; дополнительно `requests`, `replicate` (для режима WAN), `open-clip-torch` (CLIP re-ID).

> **PyTorch должен быть установлен заранее!**
> - RTX 5060/5070 (Blackwell): `pip install torch==2.7.0 torchvision --index-url https://download.pytorch.org/whl/cu128`
> - RTX 3060/4060: `pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121`


In [ ]:
import subprocess, sys, os

assert os.path.exists('src/detector.py'), (
    'Запусти ноутбук из корня video_changer/\n'
    'cd /path/to/video_changer && jupyter notebook'
)
REPO_ROOT = os.path.abspath('.')
print(f'✅ Корень репозитория: {REPO_ROOT}')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'requests', 'replicate', 'open-clip-torch'])

for p in [REPO_ROOT, os.path.join(REPO_ROOT, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

raft_core = os.path.join(REPO_ROOT, 'RAFT', 'core')
if os.path.exists(raft_core) and raft_core not in sys.path:
    sys.path.insert(0, raft_core)
    print('✅ RAFT/core добавлен в sys.path')

print('✅ Зависимости установлены')


## 2. Конфигурация

- **`API_MODE = 'wan'`** — диффузия через **Replicate** (видео WAN inpaint, см. `wan_replicate_model`). Нужен **`REPLICATE_API_TOKEN`**.
- **`API_MODE = 'nano'`** — **Gen-API Nano Banana 2**. Нужен **`GEN_API_KEY`**.

In [ ]:
import os
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Dict, Literal

@dataclass
class Config:

    input_video:      str = 'videos/input.mp4'
    output_video:     str = 'videos/output_v3.mp4'
    output_frames_dir:str = 'videos/output_frames/'

    sam2_checkpoint:  str = 'sam2_hiera_large.pt'
    sam2_config:      str = 'configs/sam2.1/sam2.1_hiera_l.yaml'
    gdino_checkpoint: str = 'groundingdino_swint_ogc.pth'
    gdino_config:     str = ''
    raft_model:       str = 'RAFT/models/raft-things.pth'

    detect_prompt:    str = 'mug'
    replace_prompt:   str = 'ceramic mug with space galaxy design, photorealistic, studio lighting'
    negative_prompt:  str = 'blurry, deformed, ugly, low quality, text, watermark'

    # Режим диффузии: wan = Replicate (видео WAN), nano = Gen-API Nano Banana 2
    API_MODE: Literal['wan', 'nano'] = 'nano'

    # Gen-API Nano Banana 2
    gen_api_key:      str = os.environ.get('GEN_API_KEY', '').strip()
    nano_resolution:  str = '0.5K'
    nano_poll_interval: float = 2.0
    nano_timeout_sec: float = 600.0

    # Replicate — только для API_MODE == 'wan'
    replicate_token:  str = os.environ.get('REPLICATE_API_TOKEN', '').strip()
    wan_replicate_model: str = 'andreasjansson/wan-1.3b-inpaint'
    wan_sampling_steps: int = 30

    box_threshold:    float = 0.35
    text_threshold:   float = 0.25
    scan_frames:      int   = 30

    fill_gaps:        int   = 2

    HIST_THRESHOLD:   float = 0.40
    CUT_IOU_THRESHOLD:float = 0.20

    FLOW_THRESHOLD:   float = 15.0
    IOU_THRESHOLD:    float = 0.50

    REID_THRESHOLD:   float = 0.75
    MAX_FRAMES_MISS:  int   = 60

    BBOX_PADDING:     float = 0.15

    feather_radius:   int   = 12
    use_poisson:      bool  = True
    blend_alpha:      float = 0.85

    device:           str   = 'cuda'
    max_frames: Optional[int] = None
    target_fps: Optional[float] = None
    save_debug_frames: bool = True

    seed:             int   = 42

cfg = Config()

os.makedirs('videos', exist_ok=True)
os.makedirs(cfg.output_frames_dir, exist_ok=True)

print(f'✅ Конфигурация:')
print(f'   Детектируем:  "{cfg.detect_prompt}"')
print(f'   Заменяем на:  "{cfg.replace_prompt[:55]}..."')
print(f'   API_MODE:      {cfg.API_MODE}  (wan=Replicate WAN, nano=Gen-API Nano Banana 2)')
if cfg.API_MODE == 'wan':
    print(f'   Replicate:     модель {cfg.wan_replicate_model}')
    print(f'   REPLICATE_TOKEN: {"задан" if cfg.replicate_token else "ПУСТО"}')
else:
    print(f'   GEN_API_KEY:   {"задан" if cfg.gen_api_key else "ПУСТО"}')
print(f'   FLOW_THR:     {cfg.FLOW_THRESHOLD} px   IOU_THR: {cfg.IOU_THRESHOLD}')


## 3. Импорты, модели из src/, VideoState

In [ ]:
import cv2, numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
import base64, io, os, sys, urllib.request, tempfile
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Dict
from PIL import Image
from tqdm import tqdm

from v3_notebook_compat import (
    load_models,
    detect_object_gdino,
    load_sam2_video_pair,
    track_object_video,
    resolve_model_path,
)
from nano_genapi import call_inpaint_crop
from replicate_wan import call_wan_inpaint_replicate

try:
    from raft_load import load_raft, compute_flow_raft as raft_compute_flow
    RAFT_AVAILABLE = True
    print('✅ raft_load.py найден — RAFT при наличии весов')
except ImportError:
    RAFT_AVAILABLE = False
    raft_compute_flow = None
    print('⚠️  raft_load недоступен — Farneback')

@dataclass
class VideoState:
    # Промежуточное состояние пайплайна

    frames:       List[np.ndarray]           = field(default_factory=list)
    masks:        List[Optional[np.ndarray]] = field(default_factory=list)
    bboxes:       List[Optional[Tuple]]      = field(default_factory=list)
    flows:        List[Optional[np.ndarray]] = field(default_factory=list)
    output_frames:List[np.ndarray]           = field(default_factory=list)

    generated: Dict[int, Tuple[np.ndarray, Tuple]] = field(default_factory=dict)

    shot_boundaries: List[int]  = field(default_factory=list)
    object_visible:  List[bool] = field(default_factory=list)

    last_appearance: Optional[torch.Tensor] = None

    fps:        float          = 25.0
    frame_size: Tuple[int,int] = (1280, 720)

print('✅ Импорты выполнены')


## 4. Загрузка видео

In [ ]:
def load_video(
    path: str,
    max_frames: Optional[int] = None
) -> Tuple[List[np.ndarray], float, Tuple[int,int]]:
    """Загружает видео → (frames BGR, fps, (w, h))."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise FileNotFoundError(f'Не могу открыть: {path}')

    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f'📹 {Path(path).name}: {total} кадров, {fps:.1f} fps, {w}×{h}')

    frames, limit = [], max_frames or total
    for _ in tqdm(range(limit), desc='Загружаем кадры'):
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()

    print(f'✅ Загружено {len(frames)} кадров')
    return frames, fps, (w, h)


def save_frames(frames: List[np.ndarray], directory: str):
    """Сохраняет кадры в папку для отладки."""
    os.makedirs(directory, exist_ok=True)
    for i, frame in enumerate(frames):
        cv2.imwrite(os.path.join(directory, f'frame_{i:05d}.jpg'), frame)


state = VideoState()
state.frames, state.fps, state.frame_size = load_video(cfg.input_video, cfg.max_frames)

## 5. Shot Cut Detection

Предварительный проход по всему видео — находим все смены камеры.
Используем гистограммы HSV: быстро и надёжно без тяжёлых моделей.

In [ ]:
def histogram_diff(f1: np.ndarray, f2: np.ndarray) -> float:
    """Несхожесть гистограмм HSV: 0.0=одинаково, 1.0=полностью разные."""
    h1 = cv2.cvtColor(f1, cv2.COLOR_BGR2HSV)
    h2 = cv2.cvtColor(f2, cv2.COLOR_BGR2HSV)
    scores = []
    for ch in range(3):
        hist1 = cv2.calcHist([h1], [ch], None, [64], [0, 256])
        hist2 = cv2.calcHist([h2], [ch], None, [64], [0, 256])
        cv2.normalize(hist1, hist1)
        cv2.normalize(hist2, hist2)
        # correlation: 1.0=идентично → несхожесть = 1 - correlation
        scores.append(cv2.compareHist(hist1, hist2, cv2.HISTCMP_CORREL))
    return 1.0 - float(np.mean(scores))


def mask_iou(m1: Optional[np.ndarray], m2: Optional[np.ndarray]) -> float:
    """IoU двух бинарных масок. None → 0.0."""
    if m1 is None or m2 is None:
        return 0.0
    a, b = m1.astype(bool), m2.astype(bool)
    inter = (a & b).sum()
    union = (a | b).sum()
    return float(inter / union) if union > 0 else 0.0


def find_shot_boundaries(
    frames: List[np.ndarray],
    hist_threshold: float
) -> List[int]:
    """
    Находит все смены камеры по разнице гистограмм.
    Первый кадр [0] всегда включён как начало первой сцены.
    """
    boundaries = [0]
    for i in tqdm(range(1, len(frames)), desc='Shot cut detection'):
        if histogram_diff(frames[i-1], frames[i]) > hist_threshold:
            boundaries.append(i)
    print(f'🎬 Найдено {len(boundaries)} сцен(а): {boundaries}')
    return boundaries


state.shot_boundaries = find_shot_boundaries(state.frames, cfg.HIST_THRESHOLD)

## 6. Загрузка моделей (GroundingDINO, SAM2, RAFT)

Используем `detector.py`, `sam_video.py`, `raft_load.py` из `src/` как есть.

In [ ]:
REPO_ROOT = os.path.abspath('.')

sam2_ckpt = resolve_model_path(REPO_ROOT, cfg.sam2_checkpoint)
gdino_ckpt = resolve_model_path(REPO_ROOT, cfg.gdino_checkpoint)

if not os.path.isfile(sam2_ckpt):
    raise FileNotFoundError(
        f'SAM2 веса не найдены: {sam2_ckpt}\n'
        f'Положите sam2_hiera_large.pt в папку models/ (как в video_changer_colab.ipynb).'
    )
if not os.path.isfile(gdino_ckpt):
    raise FileNotFoundError(
        f'Grounding DINO не найден: {gdino_ckpt}\n'
        f'Положите groundingdino_swint_ogc.pth в папку models/.'
    )

gdino_model, gdino_processor = load_models(
    gdino_config=cfg.gdino_config,
    gdino_checkpoint=gdino_ckpt,
    device=cfg.device
)

sam2_predictor, sam2_image_predictor = load_sam2_video_pair(
    checkpoint=sam2_ckpt,
    config=cfg.sam2_config,
    device=cfg.device
)

raft_model = None
if RAFT_AVAILABLE and os.path.exists(cfg.raft_model):
    raft_model = load_raft(cfg.raft_model, cfg.device)
    print(f'✅ RAFT загружен: {cfg.raft_model}')
elif RAFT_AVAILABLE:
    print(f'⚠️  RAFT веса не найдены: {cfg.raft_model} → Farneback')

print('✅ Локальные модели загружены (DINO, SAM2)')


## 7. GroundingDINO + SAM2 — детекция и трекинг масок

Для каждой сцены:
1. Сканируем `scan_frames` равномерных кадров → берём bbox с **максимальным confidence** (как в `detector.py`)
2. SAM2 Video трекает маску по всей сцене
3. Заполняем пробелы до `fill_gaps` кадров

In [ ]:
def detect_best_bbox_in_scene(
    frames: List[np.ndarray],
    start: int, end: int,
    prompt: str,
    scan_frames: int,
    box_threshold: float,
    text_threshold: float
) -> Optional[Tuple]:
    n = end - start
    indices = np.linspace(start, end - 1, min(scan_frames, n), dtype=int).tolist()

    best_bbox, best_conf = None, 0.0
    for idx in indices:
        result = detect_object_gdino(
            gdino_model, gdino_processor,
            frames[idx], prompt,
            box_threshold=box_threshold,
            text_threshold=text_threshold,
            device=cfg.device
        )
        if result is not None:
            bbox, conf = result
            if conf > best_conf:
                best_conf = conf
                best_bbox = bbox

    if best_bbox:
        print(f'    ✅ bbox={best_bbox} conf={best_conf:.3f}')
    return best_bbox


def track_all_scenes(
    frames: List[np.ndarray],
    shot_boundaries: List[int],
    cfg: Config
) -> Tuple[List, List]:
    n = len(frames)
    all_masks  = [None] * n
    all_bboxes = [None] * n
    boundaries = sorted(shot_boundaries)

    for scene_i, scene_start in enumerate(tqdm(boundaries, desc='Сцены')):
        scene_end = boundaries[scene_i + 1] if scene_i + 1 < len(boundaries) else n
        print(f'\n  Сцена {scene_i+1}/{len(boundaries)}: кадры [{scene_start}:{scene_end}]')

        bbox = detect_best_bbox_in_scene(
            frames, scene_start, scene_end,
            cfg.detect_prompt, cfg.scan_frames,
            cfg.box_threshold, cfg.text_threshold
        )
        if bbox is None:
            print(f'    ⚠️  объект не найден в сцене, пропускаем')
            continue

        scene_masks, scene_bboxes = track_object_video(
            sam2_predictor,
            sam2_image_predictor,
            frames[scene_start:scene_end],
            bbox,
            fill_gaps=cfg.fill_gaps
        )
        for i, (m, b) in enumerate(zip(scene_masks, scene_bboxes)):
            all_masks [scene_start + i] = m
            all_bboxes[scene_start + i] = b

    n_vis = sum(1 for m in all_masks if m is not None)
    print(f'\n✅ Трекинг завершён: объект найден в {n_vis}/{n} кадрах')
    return all_masks, all_bboxes


state.masks, state.bboxes = track_all_scenes(state.frames, state.shot_boundaries, cfg)


## 8. CLIP Re-Identification

Когда объект пропадает и появляется снова — CLIP cosine similarity подтверждает
что это тот же объект. При подтверждении → принудительная регенерация.

In [ ]:
import open_clip

def load_clip_reid(device: str):
    model, _, preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='laion2b_s34b_b79k'
    )
    model = model.to(device).eval()
    print('✅ CLIP ViT-B-32 загружен для re-ID')
    return model, preprocess


def get_crop_embedding(
    clip_model, preprocess,
    frame: np.ndarray, bbox: Tuple, device: str
) -> Optional[torch.Tensor]:
    """Нормализованный CLIP эмбеддинг crop'а bbox из кадра."""
    x1, y1, x2, y2 = bbox
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    pil = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    t = preprocess(pil).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = clip_model.encode_image(t)
        emb = F.normalize(emb, dim=-1)
    return emb


def is_same_object_clip(
    clip_model, preprocess,
    frame: np.ndarray, bbox: Tuple,
    ref_emb: torch.Tensor,
    threshold: float, device: str
) -> bool:
    """True если CLIP cosine similarity с reference ≥ threshold."""
    emb = get_crop_embedding(clip_model, preprocess, frame, bbox, device)
    if emb is None:
        return False
    sim = float((ref_emb * emb).sum())
    print(f'    Re-ID CLIP sim={sim:.3f} (порог {threshold})')
    return sim >= threshold


clip_model, clip_preprocess = load_clip_reid(cfg.device)

## 9. Optical Flow

Используем `raft_load.py` из `src/` (если RAFT доступен), иначе OpenCV Farneback.
Flow вычисляется **для каждого кадра** и используется для:
1. Решения `should_regenerate` — magnitude flow в области маски
2. **Варпинга** сгенерированного crop между keyframe'ами

In [ ]:
def get_optical_flow(
    frame1: np.ndarray,
    frame2: np.ndarray,
    device: str
) -> np.ndarray:
    if raft_model is not None:
        f1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB)
        f2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2RGB)
        return raft_compute_flow(raft_model, f1, f2, device)
    g1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY).astype(np.float32)
    g2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY).astype(np.float32)
    return cv2.calcOpticalFlowFarneback(
        g1, g2, None,
        pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0
    )


def flow_magnitude_in_mask(flow: np.ndarray, mask: np.ndarray) -> float:
    region = mask > 0
    if region.sum() == 0:
        return 0.0
    dx, dy = flow[..., 0][region], flow[..., 1][region]
    return float(np.sqrt(dx**2 + dy**2).mean())


def warp_with_flow(image: np.ndarray, flow: np.ndarray) -> np.ndarray:
    h, w = flow.shape[:2]
    gy, gx = np.mgrid[0:h, 0:w].astype(np.float32)
    map_x = (gx + flow[..., 0]).astype(np.float32)
    map_y = (gy + flow[..., 1]).astype(np.float32)
    return cv2.remap(image, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


print('✅ Optical flow функции готовы')
print(f'   Метод: {"RAFT" if raft_model is not None else "OpenCV Farneback"}')


## 10. Логика: regenerate vs warp

Для каждого кадра решаем: вызывать API или варпить локально.

**Всегда regenerate:** первый кадр, shot cut, объект вернулся (re-ID)  
**Conditional regenerate:** flow mag > FLOW_THRESHOLD ИЛИ mask IoU < IOU_THRESHOLD  
**Warp:** всё остальное — 0 API calls

In [ ]:
def should_regenerate(
    prev_mask: Optional[np.ndarray],
    curr_mask: Optional[np.ndarray],
    flow: Optional[np.ndarray],
    flow_thr: float,
    iou_thr: float
) -> Tuple[bool, str]:
    """
    Возвращает (нужна_регенерация, причина_строкой).
    Проверяет: flow magnitude в области маски, IoU масок.
    """
    if prev_mask is None:
        return True, 'no_prev_mask'
    if curr_mask is None:
        return False, 'object_invisible'

    # Optical flow magnitude в области объекта
    if flow is not None:
        mag = flow_magnitude_in_mask(flow, curr_mask)
        if mag > flow_thr:
            return True, f'flow={mag:.1f}px>thr'

    # IoU масок между кадрами
    iou = mask_iou(prev_mask, curr_mask)
    if iou < iou_thr:
        return True, f'iou={iou:.2f}<thr'

    return False, 'warp'


print('✅ Логика should_regenerate определена')

## 11. API диффузии: Replicate WAN или Gen-API Nano Banana 2

- **`API_MODE == 'wan'`** — клиент **Replicate**, модель по умолчанию **andreasjansson/wan-1.3b-inpaint** (видео inpaint по crop). Токен: **`REPLICATE_API_TOKEN`** или `cfg.replicate_token`.
- **`API_MODE == 'nano'`** — **Gen-API** Nano Banana 2 (как основной ноутбук). Ключ: **`GEN_API_KEY`** / `cfg.gen_api_key`.

Общие утилиты: `make_padded_bbox`, затем вызов выбранного бэкенда.


In [ ]:
def make_padded_bbox(
    bbox: Tuple, frame_shape: Tuple, padding: float
) -> Tuple:
    x1, y1, x2, y2 = bbox
    h, w = frame_shape[:2]
    pw = int((x2 - x1) * padding)
    ph = int((y2 - y1) * padding)
    return (max(0, x1-pw), max(0, y1-ph), min(w, x2+pw), min(h, y2+ph))


def call_nano_inpainting(
    frame: np.ndarray,
    mask: np.ndarray,
    bbox: Tuple,
    cfg: Config,
    frame_seed: int,
) -> Tuple[np.ndarray, Tuple]:
    pb = make_padded_bbox(bbox, frame.shape, cfg.BBOX_PADDING)
    return call_inpaint_crop(
        frame, mask, pb,
        api_key=cfg.gen_api_key,
        new_object_prompt=cfg.replace_prompt,
        old_object_prompt=cfg.detect_prompt,
        seed=frame_seed,
        resolution=cfg.nano_resolution,
        poll_interval=cfg.nano_poll_interval,
        timeout_sec=cfg.nano_timeout_sec,
    )


def call_wan_inpainting(
    frame: np.ndarray,
    mask: np.ndarray,
    bbox: Tuple,
    cfg: Config,
    frame_seed: int,
) -> Tuple[np.ndarray, Tuple]:
    if not cfg.replicate_token:
        raise RuntimeError('Для WAN задай cfg.replicate_token или REPLICATE_API_TOKEN')
    import os as _os
    _os.environ['REPLICATE_API_TOKEN'] = cfg.replicate_token
    pb = make_padded_bbox(bbox, frame.shape, cfg.BBOX_PADDING)
    prompt = f"Replace {cfg.detect_prompt} with {cfg.replace_prompt}. Photorealistic, match scene lighting."
    return call_wan_inpaint_replicate(
        frame, mask, pb,
        prompt=prompt,
        negative_prompt=cfg.negative_prompt,
        model=cfg.wan_replicate_model,
        seed=int(frame_seed),
        sampling_steps=cfg.wan_sampling_steps,
    )


def call_diffusion_inpainting(
    frame: np.ndarray,
    mask: np.ndarray,
    bbox: Tuple,
    cfg: Config,
    frame_seed: int,
) -> Tuple[np.ndarray, Tuple]:
    if cfg.API_MODE == 'wan':
        return call_wan_inpainting(frame, mask, bbox, cfg, frame_seed)
    return call_nano_inpainting(frame, mask, bbox, cfg, frame_seed)


print('✅ API: WAN (Replicate) / Nano (Gen-API) готовы')


## 12. Compositing

Вставляем crop обратно в кадр. Используем те же техники что в `main.py`:
- **Poisson seamless cloning** для бесшовного стыка (`cv2.seamlessClone`)
- **Gaussian feathering** как fallback если Poisson упал
- **Temporal blending** с предыдущим кадром для устранения мерцания

In [ ]:
def composite_frame(
    orig: np.ndarray,
    gen_crop: np.ndarray,
    mask: np.ndarray,
    pb: Tuple,
    feather_r: int = 12,
    use_poisson: bool = True
) -> np.ndarray:
    """
    Вставляет gen_crop обратно в orig по координатам padded_bbox.

    feather_r:   Gaussian blur radius краёв маски (аналог --feather-radius)
    use_poisson: Poisson seamless cloning (аналог поведения без --no-poisson)
                 Если Poisson падает — автоматический fallback на alpha blend.
    """
    result = orig.copy()
    x1, y1, x2, y2 = pb
    h_crop, w_crop = y2 - y1, x2 - x1

    if gen_crop.shape[:2] != (h_crop, w_crop):
        gen_crop = cv2.resize(gen_crop, (w_crop, h_crop))

    mask_crop = mask[y1:y2, x1:x2]

    poisson_ok = False
    if use_poisson:
        try:
            # Poisson требует маску где нарисован контур заливки
            poisson_mask = np.zeros((h_crop, w_crop), dtype=np.uint8)
            ys, xs = np.where(mask_crop > 0)
            if len(xs) > 4:  # нужно хотя бы несколько пикселей
                pts = np.stack([xs, ys], axis=1).reshape(-1, 1, 2).astype(np.int32)
                cv2.fillPoly(poisson_mask, [pts], 255)
                center = (x1 + w_crop // 2, y1 + h_crop // 2)
                result = cv2.seamlessClone(
                    gen_crop, orig, poisson_mask, center, cv2.NORMAL_CLONE
                )
                poisson_ok = True
        except cv2.error:
            pass  # fallback ниже

    if not poisson_ok:
        # Gaussian feathering alpha blend
        k = feather_r * 2 + 1
        alpha = cv2.GaussianBlur(
            mask_crop.astype(np.float32) / 255.0, (k, k), 0
        )[:, :, np.newaxis]
        orig_crop  = orig[y1:y2, x1:x2].astype(np.float32)
        blended    = alpha * gen_crop.astype(np.float32) + (1 - alpha) * orig_crop
        result[y1:y2, x1:x2] = blended.astype(np.uint8)

    return result


def temporal_blend(
    curr: np.ndarray,
    prev: Optional[np.ndarray],
    alpha: float
) -> np.ndarray:
    """
    Смешивает текущий кадр с предыдущим для подавления мерцания.
    alpha=0.85 → 85% текущий + 15% предыдущий (аналог --blend-alpha).
    """
    if prev is None or alpha >= 1.0:
        return curr
    return cv2.addWeighted(curr, alpha, prev, 1.0 - alpha, 0)


print('✅ Compositing функции готовы')

## 13. Главный пайплайн

`API_MODE`: **wan** → Replicate WAN; **nano** → Nano Banana 2.  
Далее: optical flow → regen/warp → выбранный API → composite → temporal blend.


In [ ]:
def run_pipeline_inner(state: VideoState, cfg: Config) -> VideoState:
    n         = len(state.frames)
    shots_set = set(state.shot_boundaries)
    api_label = 'WAN (Replicate)' if cfg.API_MODE == 'wan' else 'Nano Banana 2'

    prev_mask      = None
    prev_gen_crop  = None
    prev_pb        = None
    prev_out_frame = None
    was_visible    = False
    frames_missing = 0
    api_calls      = 0

    state.flows = []

    for i in tqdm(range(n), desc=f'Пайплайн ({cfg.API_MODE})'):
        frame     = state.frames[i]
        curr_mask = state.masks[i]
        curr_bbox = state.bboxes[i]
        is_cut    = (i in shots_set) and (i > 0)
        visible   = curr_mask is not None and curr_bbox is not None

        state.object_visible.append(visible)

        if not visible:
            state.output_frames.append(frame.copy())
            state.flows.append(None)
            frames_missing += 1
            was_visible = False
            if frames_missing > cfg.MAX_FRAMES_MISS:
                prev_mask = None; prev_gen_crop = None
                state.last_appearance = None
                print(f'  ⚠️  [{i}] объект пропал на {frames_missing} кадров → сброс')
            continue

        reappeared = (not was_visible) and (frames_missing > 0)
        frames_missing = 0
        was_visible = True

        force_regen = False
        if reappeared and state.last_appearance is not None:
            confirmed = is_same_object_clip(
                clip_model, clip_preprocess,
                frame, curr_bbox, state.last_appearance,
                cfg.REID_THRESHOLD, cfg.device
            )
            if confirmed:
                print(f'  🔁 [{i}] re-ID подтверждён → принудит. регенерация')
                force_regen = True

        emb = get_crop_embedding(clip_model, clip_preprocess, frame, curr_bbox, cfg.device)
        if emb is not None:
            state.last_appearance = emb

        flow = None
        if i > 0 and not is_cut:
            flow = get_optical_flow(state.frames[i-1], frame, cfg.device)
        state.flows.append(flow)

        need_regen, reason = should_regenerate(
            prev_mask, curr_mask, flow, cfg.FLOW_THRESHOLD, cfg.IOU_THRESHOLD
        )
        need_regen = need_regen or is_cut or force_regen or (i == 0)

        if need_regen:
            tag = 'cut' if is_cut else ('reid' if force_regen else reason)
            print(f'  🔄 [{i}] {api_label} [{tag}]')
            frame_seed = int(cfg.seed) + i
            gen_crop, pb = call_diffusion_inpainting(frame, curr_mask, curr_bbox, cfg, frame_seed)
            state.generated[i] = (gen_crop, pb)
            api_calls += 1
        else:
            if prev_gen_crop is not None and flow is not None and prev_pb is not None:
                px1, py1, px2, py2 = prev_pb
                fh, fw = flow.shape[:2]
                if py2 <= fh and px2 <= fw and py1 < py2 and px1 < px2:
                    gen_crop = warp_with_flow(prev_gen_crop, flow[py1:py2, px1:px2])
                else:
                    gen_crop = prev_gen_crop
                pb = prev_pb
            else:
                gen_crop, pb = prev_gen_crop, prev_pb

        if gen_crop is not None and pb is not None:
            out = composite_frame(
                frame, gen_crop, curr_mask, pb,
                feather_r=cfg.feather_radius,
                use_poisson=cfg.use_poisson
            )
            out = temporal_blend(out, prev_out_frame, cfg.blend_alpha)
        else:
            out = frame.copy()

        state.output_frames.append(out)
        prev_mask = curr_mask; prev_gen_crop = gen_crop
        prev_pb = pb; prev_out_frame = out

    n_vis = sum(state.object_visible)
    print(f'\n✅ Пайплайн ({cfg.API_MODE}) завершён')
    print(f'   API вызовов: {api_calls} / {n_vis} кадров с объектом')
    print(f'   Warp:        {max(0, n_vis - api_calls)} кадров (без API)')
    return state


def run_pipeline(state: VideoState, cfg: Config) -> VideoState:
    print(f'🚀 Запуск: API_MODE={cfg.API_MODE}')
    return run_pipeline_inner(state, cfg)


print('✅ Пайплайн определён')


## 14. Запуск

In [ ]:
if cfg.API_MODE == 'wan':
    assert cfg.replicate_token, (
        '❌ Для WAN задай REPLICATE_API_TOKEN или cfg.replicate_token в конфиге.'
    )
else:
    assert cfg.gen_api_key, (
        '❌ Для Nano Banana задай GEN_API_KEY или cfg.gen_api_key в конфиге.'
    )

state = run_pipeline(state, cfg)

print(f'\nОбработано кадров:  {len(state.output_frames)}')
print(f'Кадров с объектом:  {sum(state.object_visible)}')
print(f'API вызовов:        {len(state.generated)}')
print(f'Shot cuts:          {state.shot_boundaries}')


## 15. Сборка видео + сохранение кадров

In [ ]:
import imageio

def assemble_video(
    frames: List[np.ndarray],
    output_path: str,
    fps: float,
    frame_size: Tuple[int,int]
):
    """
    Собирает видео через imageio + ffmpeg (как в проекте).
    frame_size: (width, height).
    """
    writer = imageio.get_writer(
        output_path, fps=fps,
        codec='libx264', quality=8, pixelformat='yuv420p'
    )
    for frame in tqdm(frames, desc='Записываем видео'):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        if (rgb.shape[1], rgb.shape[0]) != frame_size:
            rgb = cv2.resize(rgb, frame_size)
        writer.append_data(rgb)
    writer.close()

    size_mb = os.path.getsize(output_path) / 1024 / 1024
    print(f'✅ Видео: {output_path} ({size_mb:.1f} MB)')


assemble_video(
    state.output_frames,
    cfg.output_video,
    fps=cfg.target_fps or state.fps,
    frame_size=state.frame_size
)

if cfg.save_debug_frames:
    save_frames(state.output_frames, cfg.output_frames_dir)
    print(f'✅ Кадры сохранены: {cfg.output_frames_dir}')

## 16. Визуализация и статистика

In [ ]:
def visualize_flow(flow: np.ndarray) -> np.ndarray:
    """Optical flow → HSV цветовая карта (RGB для matplotlib)."""
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros((*flow.shape[:2], 3), dtype=np.uint8)
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 1] = 255
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def show_comparison(
    state: VideoState,
    indices: List[int],
    show_flow: bool = True
):
    """Оригинал / маска+bbox / результат / optical flow для заданных кадров."""
    ncols = 4 if show_flow else 3
    fig, axes = plt.subplots(len(indices), ncols, figsize=(5*ncols, 4*len(indices)))
    if len(indices) == 1:
        axes = [axes]

    for row, idx in enumerate(indices):
        frame = state.frames[idx]
        mask  = state.masks[idx]  if idx < len(state.masks)  else None
        bbox  = state.bboxes[idx] if idx < len(state.bboxes) else None
        out   = state.output_frames[idx] if idx < len(state.output_frames) else frame
        flow  = state.flows[idx]  if idx < len(state.flows)  else None

        # Оригинал
        axes[row][0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        title = f'[{idx}] Оригинал'
        if idx in state.shot_boundaries: title += ' 🎬CUT'
        axes[row][0].set_title(title); axes[row][0].axis('off')

        # Маска
        overlay = frame.copy()
        if mask is not None: overlay[mask > 0] = (0, 200, 0)
        if bbox is not None: cv2.rectangle(overlay, bbox[:2], bbox[2:], (0, 200, 0), 2)
        axes[row][1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[row][1].set_title('SAM2 маска'); axes[row][1].axis('off')

        # Результат
        regen_tag = '🔄API' if idx in state.generated else '↩warp'
        axes[row][2].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
        axes[row][2].set_title(f'Результат {regen_tag}'); axes[row][2].axis('off')

        # Optical flow
        if show_flow:
            if flow is not None:
                axes[row][3].imshow(visualize_flow(flow))
                axes[row][3].set_title('Optical Flow')
            else:
                axes[row][3].text(0.5, 0.5, 'нет flow\n(cut / первый кадр)',
                                  ha='center', va='center',
                                  transform=axes[row][3].transAxes)
            axes[row][3].axis('off')

    plt.tight_layout()
    plt.show()


def show_shot_boundaries(state: VideoState):
    """Показывает кадры вокруг каждого shot cut."""
    for cut in state.shot_boundaries[1:]:
        print(f'\n--- Shot cut @ кадр {cut} ---')
        ctx = [max(0, cut-1), cut, min(len(state.frames)-1, cut+1)]
        show_comparison(state, ctx, show_flow=False)


def print_pipeline_stats(state: VideoState, cfg: Config):
    """Итоговая статистика."""
    n       = len(state.frames)
    n_vis   = sum(state.object_visible)
    n_regen = len(state.generated)
    n_warp  = max(0, n_vis - n_regen)
    n_cuts  = len(state.shot_boundaries) - 1

    print('=' * 55)
    print('📊  СТАТИСТИКА ПАЙПЛАЙНА')
    print('=' * 55)
    print(f'Режим API:              {cfg.API_MODE.upper()}')
    print(f'Всего кадров:           {n}')
    print(f'Кадров с объектом:      {n_vis}  ({100*n_vis/max(n,1):.1f}%)')
    print(f'Shot cuts:              {n_cuts}')
    print(f'API вызовов (regen):    {n_regen}')
    print(f'Warp кадров (локально): {n_warp}')
    if n_vis > 0:
        print(f'API % от видимых:       {100*n_regen/n_vis:.1f}%')
    print('---')
    print(f'FLOW_THRESHOLD:         {cfg.FLOW_THRESHOLD} px')
    print(f'IOU_THRESHOLD:          {cfg.IOU_THRESHOLD}')
    print(f'REID_THRESHOLD:         {cfg.REID_THRESHOLD}')
    print(f'BBOX_PADDING:           {cfg.BBOX_PADDING*100:.0f}%')
    print(f'Poisson blending:       {cfg.use_poisson}')
    print(f'Temporal blend alpha:   {cfg.blend_alpha}')
    print(f'Flow метод:             {"RAFT" if raft_model else "Farneback"}')
    print('=' * 55)


# ── Показываем результаты ─────────────────────────────────────────────────────
sample = list(range(0, min(5, len(state.output_frames))))
show_comparison(state, sample)

if len(state.shot_boundaries) > 1:
    show_shot_boundaries(state)

print_pipeline_stats(state, cfg)